# Projet Api Immo Lille — Analyse & Modélisation

## 📜 Objectifs du projet

L’objectif de ce projet est d’analyser un jeu de données de transactions immobilières à Lille en 2022,  
afin de comprendre quels facteurs influencent le prix au mètre carré des logements.  

Nous allons :  
- Charger et filtrer les données.  
- Nettoyer et préparer le dataset pour la modélisation.  
- Séparer maisons et appartements pour des analyses distinctes.  
- Construire plusieurs modèles de régression pour prédire le prix au m².  
- Comparer les performances des modèles pour choisir le meilleur.

---

## 1. Chargement des bibliothèques nécessaires

Avant toute chose, nous importons les bibliothèques indispensables pour :  
- Manipuler les données (pandas, numpy).  
- Visualiser les données (matplotlib, seaborn).  
- Construire, évaluer et optimiser les modèles (scikit-learn, xgboost).  
- Sauvegarder les modèles entraînés (joblib).

In [ ]:
#import utile pour le projet machine learning 

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 
import joblib

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.compose import ColumnTransformer

## 2. Chargement et filtrage initial des données

Nous chargeons ici le fichier CSV contenant les transactions.  

Pour garder une analyse ciblée et homogène,  
nous ne conservons que les logements ayant exactement 4 pièces principales.  

Nous sélectionnons aussi les colonnes pertinentes pour notre étude :  
surface bâtie, type de logement, surface du terrain, nombre de lots, et valeur foncière.


In [ ]:
df = pd.read_csv("../data/lille_2022.csv")

# Ne garder que les logements à 4 pièces
df = df[df["Nombre pieces principales"] == 4]

# Sélection des colonnes utiles
cols = [
    "Surface reelle bati",
    "Nombre pieces principales",
    "Type local",
    "Surface terrain",
    "Nombre de lots",
    "Valeur fonciere",
]
df = df[cols]
display(df)


## 🏠 3. Séparation Maisons / Appartements et Nettoyage

In [ ]:
# Séparer les maisons et appartements
df_maisons = df[df['Type local'].str.lower().str.contains('maison')].copy()
df_apparts = df[df['Type local'].str.lower().str.contains('appartement')].copy()

# Nettoyage des NA :
#df_maisons = df_maisons.dropna(subset=['Surface reelle bati', 'Surface terrain', 'Nombre de lots', 'Valeur fonciere'])
df_maisons['Surface terrain'] = df_maisons['Surface terrain'].fillna(0)

# Pour les appartements, on remplace le terrain manquant par 0
df_apparts['Surface terrain'] = df_apparts['Surface terrain'].fillna(0)


In [ ]:
print("Maisons:")
display(df_maisons)

print("\nAppartements:")
display(df_apparts)

## 🔍 4. Vérification des valeurs nulles et zéros

In [ ]:
def detect_nan_and_zeros(df: pd.DataFrame) -> pd.DataFrame:
    num_cols = df.select_dtypes(include='number').columns
    results = []
    for col in num_cols:
        n_nan = df[col].isna().sum()
        n_zero = (df[col] == 0).sum()
        results.append({"column": col, "n_nan": n_nan, "n_zero": n_zero})
    return pd.DataFrame(results)

print("Maisons:")
display(detect_nan_and_zeros(df_maisons))

print("\nAppartements:")
display(detect_nan_and_zeros(df_apparts))


## 🧮 5. Calcul du prix au m²

In [ ]:
df_maisons['prix_m2'] = df_maisons['Valeur fonciere'] / df_maisons['Surface reelle bati']
df_apparts['prix_m2'] = df_apparts['Valeur fonciere'] / df_apparts['Surface reelle bati']


## 📊 6. Analyse exploratoire : distribution et outliers

In [ ]:
# Boxplots prix/m2
fig, ax = plt.subplots(1, 2, figsize=(12,5))
sns.boxplot(data=df_maisons, y='prix_m2', ax=ax[0]).set_title('Maisons')
sns.boxplot(data=df_apparts, y='prix_m2', ax=ax[1]).set_title('Appartements')
plt.tight_layout()
plt.show()

# Distribution
sns.histplot(df_maisons['prix_m2'], bins=50, kde=True)
plt.title('Distribution du prix au m² (Maisons)')
plt.show()

sns.histplot(df_apparts['prix_m2'], bins=50, kde=True)
plt.title('Distribution du prix au m² (Appartements)')
plt.show()


## 🔍 7. Détection et suppression des valeurs aberrantes

In [ ]:
def detect_outliers(df, column):
    Q1, Q3 = df[column].quantile(0.25), df[column].quantile(0.75)
    IQR = Q3 - Q1
    lb, ub = Q1 - 1.5*IQR, Q3 + 1.5*IQR
    return lb, ub

# Nettoyage pour les maisons
for col in ['prix_m2', 'Surface reelle bati', 'Surface terrain']:
    lb, ub = detect_outliers(df_maisons, col)
    df_maisons = df_maisons[(df_maisons[col] >= lb) & (df_maisons[col] <= ub)]

# Nettoyage pour les appartements
for col in ['prix_m2', 'Surface reelle bati']:
    lb, ub = detect_outliers(df_apparts, col)
    df_apparts = df_apparts[(df_apparts[col] >= lb) & (df_apparts[col] <= ub)]

## 🧮 8. Sélection des features et de la cible

In [ ]:
#df_maisons=df_maisons["Surface reelle bati", "Nombre pieces principales",  "Type local", "Surface terrain", "Nombre de lots"]
#df_apparts=[]
display(df_maisons)
display(df_apparts)

## 🤖 9. Fonction d’entraînement et d’évaluation des modèles


In [ ]:
def run_all_models(df, target="prix_m2", title="Comparaison des modèles"):
    features = ["Surface reelle bati", "Nombre pieces principales",  "Type local", "Surface terrain", "Nombre de lots"]
    X = df[features]
    y = df[target]

    # Train/test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Préprocesseur : OneHotEncoder sur la variable catégorielle
    preprocessor = ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore"), ["Type local"]),
            ("num", StandardScaler(), ["Surface reelle bati", "Nombre pieces principales", "Surface terrain", "Nombre de lots"]),
        ]
    )

    # Grilles d'hyperparamètres
    param_grids = {
        "Decision Tree": {"model__max_depth": [3, 5, None], "model__min_samples_split": [2, 10]},
        "Random Forest": {"model__n_estimators": [50, 100], "model__max_depth": [3, None]},
        "XGBoost": {"model__n_estimators": [50, 100], "model__learning_rate": [0.01, 0.1]},
    }

    models = {
        "Decision Tree": DecisionTreeRegressor(random_state=42),
        "Random Forest": RandomForestRegressor(random_state=42),
        "XGBoost": XGBRegressor(random_state=42, verbosity=0),
    }

    results = {}
    best_pipelines = {}

    # Boucle sur les modèles
    for name, model in models.items():
        pipe = Pipeline([("preprocessing", preprocessor), ("model", model)])
        grid = GridSearchCV(pipe, param_grids[name], cv=5, scoring="neg_mean_squared_error", n_jobs=-1)
        grid.fit(X_train, y_train)

        best_model = grid.best_estimator_
        y_pred = best_model.predict(X_test)

        mse = mean_squared_error(y_test, y_pred)
        mae = mean_absolute_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)

        results[name] = {"MSE": mse, "MAE": mae, "R2": r2}
        best_pipelines[name] = best_model

        print(f"{name} : MSE={mse:.4f}, MAE={mae:.4f}, R2={r2:.4f}")

    # Modèle linéaire + SelectKBest
    pipe_lin = Pipeline(
        [("preprocessing", preprocessor), ("select", SelectKBest(score_func=f_regression)), ("model", LinearRegression())]
    )
    param_grid_lin = {"select__k": list(range(1, len(features) + 1))}
    grid_lin = GridSearchCV(pipe_lin, param_grid_lin, cv=5, scoring="neg_mean_squared_error", n_jobs=-1)
    grid_lin.fit(X_train, y_train)

    best_model_lin = grid_lin.best_estimator_
    y_pred_lin = best_model_lin.predict(X_test)

    mse_lin = mean_squared_error(y_test, y_pred_lin)
    mae_lin = mean_absolute_error(y_test, y_pred_lin)
    r2_lin = r2_score(y_test, y_pred_lin)

    results["Linear + SelectKBest"] = {"MSE": mse_lin, "MAE": mae_lin, "R2": r2_lin}
    best_pipelines["Linear + SelectKBest"] = best_model_lin

    print(f"Linear + SelectKBest : MSE={mse_lin:.4f}, MAE={mae_lin:.4f}, R2={r2_lin:.4f}")

    # Graphiques
    results_df = pd.DataFrame(results).T.reset_index().rename(columns={"index": "Model"})
    fig, ax = plt.subplots(1, 2, figsize=(14, 6))
    sns.barplot(data=results_df, x="Model", y="MSE", ax=ax[0], palette="Blues_d").set_title("Comparaison des MSE")
    sns.barplot(data=results_df, x="Model", y="R2", ax=ax[1], palette="Greens_d").set_title("Comparaison des R2")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

    return results_df, best_pipelines

In [ ]:
# ========================
# Fonction d'export
# ========================
def save_models(best_pipelines, prefix):
    for name, model in best_pipelines.items():
        filename = f"{prefix}_{name.replace(' ', '_').replace('+', 'plus')}.pkl"
        joblib.dump(model, filename)
        print(f"✅ Modèle {name} sauvegardé dans {filename}")

In [ ]:
target = "prix_m2"

# Lancement de l'entraînement et sauvegarde pour les maisons
print("### Modèles pour maisons ###")
results_df_maison, best_pipelines_maison = run_all_models(df_maisons, target, title="Modèles pour maisons")
save_models(best_pipelines_maison, prefix="maisons")

# Lancement de l'entraînement et sauvegarde pour les appartements
print("\n### Modèles pour appartements ###")
results_df_apparts, best_pipelines_apparts = run_all_models(df_apparts, target, title="Modèles pour appartements")
save_models(best_pipelines_apparts, prefix="apparts")